### Find patent-paper-author info by OpenAlex

In [ ]:
import pandas as pd
import glob
import yaml
from tqdm import tqdm

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

In [ ]:
import os
os.chdir('../../')

In [ ]:
def regex_search_match(input_str):
    import re
    output = re.search('^(CN-.*)', input_str, re.IGNORECASE)
    return output.group(1).replace('-', '').upper() if output else None

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
ROS_patent_paper = pd.read_csv(dataset_config['path_pcs'] + '_pcs_oa.csv', usecols=['oaid', 'patent'], engine='pyarrow').rename(columns={'oaid': 'work_id', 'patent': 'patent_id'})
ROS_patent_paper

In [ ]:
# Select China pantent cited papers
ROS_patent_paper['patent_id'] = ROS_patent_paper['patent_id'].parallel_apply(regex_search_match)
ROS_patent_paper = ROS_patent_paper.dropna().rename(columns={'work_id': 'paperid'})
ROS_patent_paper

In [ ]:
# Clean the data by removing rows with missing values and duplicates.
patent_paper_data = ROS_patent_paper.dropna().drop_duplicates()
patent_paper_data

In [ ]:
# Create a list of all .csv.gz files in the directory
file_pattern = os.path.join(dataset_config['path_openalex'], 'works_authorships_*.csv.gz')
csv_files = glob.glob(file_pattern)

# Read each file into a DataFrame and store them in a list
dataframes = []
for file in tqdm(csv_files):
    try:
        df = pd.read_csv(file, compression='gzip')
        df['author_position'] = df['author_position'].apply(lambda x: {'first': 1, 'middle': 0, 'last': -1}[x]).astype(pd.Int8Dtype())
        df['author_id'] = df['author_id'].astype(pd.Int64Dtype())
        df['institution_id'] = df['institution_id'].astype(pd.Int64Dtype())
        if len(df) >= 1:
            dataframes.append(df.rename(columns={'work_id': 'paperid', 'country': 'ctry'}))
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
oa_data = pd.concat(dataframes, ignore_index=True)
oa_data

In [ ]:
del dataframes

In [ ]:
# Merge patent-paper data with the MAG affiliation data on the 'paperid' column.
# A left join ensures all patent-paper pairs are kept even if they lack affiliation data.
relevant_data = patent_paper_data.merge(oa_data, on='paperid', how='left')
relevant_data

In [ ]:
relevant_data.dtypes

In [ ]:
# Filter the merged data to only include rows without missing values (valid matches).
relevant_data_valid = relevant_data.dropna(subset=['institution_id', 'affiliation', 'ctry'], how='all').drop_duplicates()
relevant_data_valid

### Extract non-matching entires

In [ ]:
relevant_data_missing = relevant_data[relevant_data.author_id.isna()].drop_duplicates()
relevant_data_missing

In [ ]:
paper_year = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/OA1_paper_year.parquet')
paper_year

In [ ]:
oa_matches = relevant_data_missing[['paperid', 'patent_id']].merge(paper_year, on='paperid', how='left')
oa_matches

### Processing missing papers

In [ ]:
missing_paperids = oa_matches[oa_matches.publication_year.isna()].paperid.drop_duplicates()
missing_paperids

In [ ]:
# Create a list of all .csv.gz files in the directory
file_pattern = os.path.join(dataset_config['path_openalex'], 'merged_ids/works/*.csv.gz')
csv_files = glob.glob(file_pattern)

# Read each file into a DataFrame and store them in a list
dataframes = []
for file in tqdm(csv_files):
    try:
        df = pd.read_csv(file, compression='gzip', usecols=['id', 'merge_into_id'])
        if len(df) >= 1:
            dataframes.append(df.rename(columns={'work_id': 'paperid', 'country': 'ctry'}))
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
merged_ids = pd.concat(dataframes, ignore_index=True)
merged_ids['id'] = merged_ids['id'].str.replace(r'^W', '', regex=True).astype(pd.Int64Dtype())
merged_ids['merge_into_id'] = merged_ids['merge_into_id'].str.replace(r'^W', '', regex=True).astype(pd.Int64Dtype())
merged_ids = merged_ids.set_index('id')
merged_ids

In [ ]:
ids_mapper = []
ids_notfound = []
ids_deleted = []
DELETED_WORK = 4285719527

for missing_id in missing_paperids.to_list():
    if missing_id in merged_ids.index:
        new_id = merged_ids.loc[missing_id].merge_into_id
    else:
        ids_notfound.append(missing_id)
        continue

    while new_id in merged_ids.index:
        new_id = merged_ids.loc[new_id].merge_into_id
        #print(missing_id, new_id, sep='\t')
    
    if new_id == DELETED_WORK:
        ids_deleted.append(missing_id)
    else:
        ids_mapper.append((missing_id, new_id))

In [ ]:
len(ids_notfound), len(ids_deleted), len(ids_mapper)

In [ ]:
missing_ids_mapper = pd.DataFrame(ids_mapper, columns=['paperid', 'newid'])
missing_ids_mapper

In [ ]:
matched_missing = relevant_data_missing[['paperid', 'patent_id']].merge(missing_ids_mapper, on='paperid')
matched_missing

In [ ]:
matched_missing = matched_missing.rename(columns={'paperid': 'prev_id', 'newid': 'paperid'})
matched_missing

In [ ]:
merged_missing = matched_missing.merge(oa_data, on='paperid', how='left')
merged_missing

In [ ]:
merged_missing_valid = merged_missing.dropna(subset=['institution_id', 'affiliation', 'ctry'], how='all').drop_duplicates()
merged_missing_valid

In [ ]:
relevant_data_all = pd.concat([relevant_data_valid, merged_missing_valid[['patent_id', 'paperid', 'author_position', 'author_id', 'institution_id', 'affiliation', 'ctry']]], ignore_index=True).drop_duplicates()
relevant_data_all

In [ ]:
def check_paper_patent_matched(data):
    _paper_patent_matched = data[['paperid', 'patent_id']].drop_duplicates()
    print(f'ROS matched patent-paper pair in OpenAlex: {len(_paper_patent_matched)} / {len(patent_paper_data)} = {len(_paper_patent_matched) / len(patent_paper_data) * 100:.4f}%')

In [ ]:
check_paper_patent_matched(relevant_data_valid)
check_paper_patent_matched(merged_missing_valid)

In [ ]:
relevant_data_all.to_parquet(dataset_config['path_processed'] + 'CN_CN/OA2_relevant_data.parquet')

In [ ]:
pp_mapped = pd.concat([relevant_data_valid[['patent_id', 'paperid']], merged_missing_valid[['patent_id', 'prev_id']].rename(columns={'prev_id': 'paperid'})], ignore_index=True)
pp_mapped

In [ ]:
paper_patent_matched = pp_mapped[['paperid', 'patent_id']].drop_duplicates()
paper_patent_matched

In [ ]:
len(patent_paper_data), len(paper_patent_matched), len(patent_paper_data) - len(paper_patent_matched)

In [ ]:
patent_paper_diff = patent_paper_data.merge(paper_patent_matched, how='left', indicator=True)
patent_paper_diff

In [ ]:
patent_paper_diff = patent_paper_diff[patent_paper_diff['_merge'] == 'left_only']
patent_paper_diff

In [ ]:
patent_paper_diff[['paperid', 'patent_id']].to_parquet(dataset_config['path_processed'] + 'CN_CN/OA2_not_matched.parquet')